In [362]:
import pandas as pd
import numpy as np
import re
from ast import literal_eval

In [363]:
column_ratings = ["user_id", "movie_id_ml", "rating", "rating_timestamp"]

df_ratings = pd.read_csv('ml-100k/u.data', delimiter='\t', names=column_ratings)
df_ratings["rating_timestamp"] = pd.to_datetime(df_ratings["rating_timestamp"], unit="s")

print(df_ratings.shape)
print(df_ratings.dtypes)
df_ratings.head()

(100000, 4)
user_id                      int64
movie_id_ml                  int64
rating                       int64
rating_timestamp    datetime64[ns]
dtype: object


,user_id,movie_id_ml,rating,rating_timestamp
0,196,242,3,1997-12-04 15:55:49
1,186,302,3,1998-04-04 19:22:22
2,22,377,1,1997-11-07 07:18:36
3,244,51,2,1997-11-27 05:02:03
4,166,346,1,1998-02-02 05:33:16


#### Movies

In [364]:
def clean_movie_title(movie_title):
    if movie_title.split(" ")[-1].startswith("("):
        # remove year from the title, e.g. Toy Story (1995) --> Toy Story
        movie_title = (" ".join(movie_title.split(" ")[:-1])).strip()

    if movie_title.title().split(',')[-1].strip() in ['The', 'A']:
        # article + movie title, e.g. Saint, The --> The Saint
        movie_title = (movie_title.title().split(',')[-1].strip() + " " + " ".join(movie_title.title().split(',')[:-1])).strip()

    # otherwise, it was converting The Devil's Advocate to The Devil'S Advocate
    movie_title = movie_title.lower()
    return movie_title

In [365]:
column_item = ["movie_id_ml", "title", "release", "vrelease", "url", "unknown", 
                    "action", "adventure", "animation", "childrens", "comedy",
                   "crime", "documentary", "drama", "fantasy", "noir", "horror",
                   "musical", "mystery", "romance", "scifi", "thriller",
                   "war", "western"]

df_ML_movies = pd.read_csv('ml-100k/u.item', delimiter='|', names=column_item, encoding = "ISO-8859-1")
df_ML_movies = df_ML_movies.drop(columns=["vrelease"])
df_ML_movies["title"] = df_ML_movies["title"].apply(lambda row : clean_movie_title(row))   
df_ML_movies["release"] = df_ML_movies["release"].apply(lambda x : str(x).split("-")[-1])

# drop rows where movie starts with brackets, those are some strange names...
df_ML_movies = df_ML_movies[~df_ML_movies.title.str.startswith("(")]

# handle seven (se7en) movies, creating new rows containing the content of brackets
_df = df_ML_movies[df_ML_movies.title.str.contains("(", regex=False)]
_df.title = _df.title.apply(lambda x: re.search(r'\((.*?)\)', x).group(1).strip() if re.search(r'\((.*?)\)', x) else x.strip())
df_ML_movies = pd.concat([df_ML_movies, _df], ignore_index=True)

print(df_ML_movies.shape)
print(df_ML_movies.dtypes)
df_ML_movies.head()

(1767, 23)
movie_id_ml     int64
title          object
release        object
url            object
unknown         int64
action          int64
adventure       int64
animation       int64
childrens       int64
comedy          int64
crime           int64
documentary     int64
drama           int64
fantasy         int64
noir            int64
horror          int64
musical         int64
mystery         int64
romance         int64
scifi           int64
thriller        int64
war             int64
western         int64
dtype: object


C:\Users\ACER\AppData\Local\Temp\ipykernel_6736\2832436488.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  _df.title = _df.title.apply(lambda x: re.search(r'\((.*?)\)', x).group(1).strip() if re.search(r'\((.*?)\)', x) else x.strip())


,movie_id_ml,title,release,url,unknown,action,adventure,animation,childrens,comedy,crime,documentary,drama,fantasy,noir,horror,musical,mystery,romance,scifi,thriller,war,western
0,1,toy story,1995,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,goldeneye,1995,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
2,3,four rooms,1995,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
3,4,get shorty,1995,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
4,5,copycat,1995,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0


#### Users

In [366]:
column_user = ["user_id", "user_age", "user_gender", "user_occupation", "user_zipcode"]

df_users = pd.read_csv('ml-100k/u.user', delimiter='|', names=column_user, encoding = "ISO-8859-1")
df_users.head()

,user_id,user_age,user_gender,user_occupation,user_zipcode
0,1,24,M,technician,85711
1,2,53,F,other,94043
2,3,23,M,writer,32067
3,4,24,M,technician,43537
4,5,33,F,other,15213


# IMDb dataset

In [367]:
column_cast = ["cast_id", "person_id", "movie_id", "person_role_id", "note", "nr_order", "role_id"]

df_cast = pd.read_csv('output/cast_info.csv', delimiter=',', names=column_cast, encoding = "ISO-8859-1", low_memory=False)
df_cast['role_id'] = pd.to_numeric(df_cast['role_id'], errors='coerce')
df_cast = df_cast.drop(columns=["note", "nr_order", "person_role_id"])

print(df_cast.dtypes)
print(df_cast.shape)
df_cast.head()

cast_id       object
person_id     object
movie_id      object
role_id      float64
dtype: object
(217692, 4)


,cast_id,person_id,movie_id,role_id
0,ï»¿cast_id,person_id,movie_id,NaN
1,1,65731,19995,413.0
2,2,8691,19995,413.0
3,3,10205,19995,413.0
4,4,32747,19995,413.0


In [368]:
column_people = ["person_id", "cast_name", "imdb_idx", "imdb_id", "cast_gender", "name_cf", "name_nf", "surname", "md5"]

df_people = pd.read_csv('output/name.csv', delimiter=',', names=column_people, encoding = "ISO-8859-1", low_memory=False)

print(df_people.dtypes)
print(df_people.shape)
df_people = df_people.drop(columns=["imdb_idx", "imdb_id", "md5", "name_cf", "name_nf", "surname"])
df_people.head()

person_id      object
cast_name      object
imdb_idx       object
imdb_id        object
cast_gender    object
name_cf        object
name_nf        object
surname        object
md5            object
dtype: object
(98995, 9)


,person_id,cast_name,cast_gender
0,ï»¿person_id,cast_name,cast_gender
1,65731,Sam Worthington,2
2,8691,Zoe Saldana,1
3,10205,Sigourney Weaver,1
4,32747,Stephen Lang,2


In [369]:
import pandas as pd
import re

columns_movies = ["movie_id", "title", "imdb_idx",
                  "movie_kind", "release", "imdb_id", "phonetic", "episode_id",
                  "season", "episode", "series_years", "md5"]

# 1. THÊM header=0 ở đây để loại bỏ dòng tiêu đề trùng lặp trong file
df_IMDb_movies = pd.read_csv('output/title.csv', delimiter=',', names=columns_movies, header=0, encoding="ISO-8859-1", low_memory=False)

df_IMDb_movies = df_IMDb_movies.drop(columns=["imdb_idx", "imdb_id", "phonetic", "md5", "episode_id", "episode", "movie_kind", "season", "series_years"])
df_IMDb_movies = df_IMDb_movies.dropna(subset=['release'])

# Thay vì ép kiểu int phức tạp, ta chuyển thành string rồi cắt chuỗi lấy năm (4 ký tự cuối hoặc sau dấu gạch ngang)
df_IMDb_movies["release"] = df_IMDb_movies["release"].apply(lambda x : str(x).split("/")[-1].strip())

# we lowered in MovieLens as well
df_IMDb_movies = df_IMDb_movies.dropna(subset=["title"])
df_IMDb_movies["title"] = df_IMDb_movies["title"].apply(lambda x: x.lower())

# drop rows where movie starts with brackets, those are some strange names...
df_IMDb_movies = df_IMDb_movies[~df_IMDb_movies.title.str.startswith("(")]

# handle seven (se7en) movies, creating new rows containing the content of brackets
_df = df_IMDb_movies[df_IMDb_movies.title.str.contains("(", regex=False)].copy()
_df.title = _df.title.apply(lambda x: re.search(r'\((.*?)\)', x).group(1).strip() if re.search(r'\((.*?)\)', x) else x.strip())

# 2. SỬA .append() thành pd.concat()
df_IMDb_movies = pd.concat([df_IMDb_movies, _df], ignore_index=True)

print(df_IMDb_movies.dtypes)
print(df_IMDb_movies.shape)
df_IMDb_movies.head()

movie_id     int64
title       object
release     object
dtype: object
(4803, 3)


,movie_id,title,release
0,19995,avatar,2009
1,285,pirates of the caribbean: at world's end,2007
2,206647,spectre,2015
3,49026,the dark knight rises,2012
4,49529,john carter,2012


In [370]:
column_movie_keyword = ["mkid", "movie_id", "keyword_id"]

df_movie_keywords = pd.read_csv('output/movie_keywords.csv', delimiter=',', names=column_movie_keyword, encoding = "ISO-8859-1")
print(df_movie_keywords.dtypes)
print(df_movie_keywords.shape)
df_movie_keywords = df_movie_keywords.drop(columns=["mkid"])
df_movie_keywords.head()

mkid          object
movie_id      object
keyword_id    object
dtype: object
(36177, 3)


,movie_id,keyword_id
0,movie_id,keyword_id
1,19995,1463
2,19995,2964
3,19995,3386
4,19995,3388


In [371]:
column_movie_companies = ["mcid", "movie_id", "company_id", "ctype", "note"]

df_movie_companies = pd.read_csv('output/movie_companies.csv', delimiter=',', names=column_movie_companies, encoding = "ISO-8859-1")
print(df_movie_companies.dtypes)
print(df_movie_companies.shape)
df_movie_companies = df_movie_companies.drop(columns=["mcid", "ctype"])

df_movie_companies.head()

mcid          object
movie_id      object
company_id    object
ctype         object
note          object
dtype: object
(13673, 5)


,movie_id,company_id,note
0,movie_id,company_id,note
1,19995,289,NaN
2,19995,306,NaN
3,19995,444,NaN
4,19995,574,NaN


In [372]:
column_companies = ["company_id", "name", "country", "imdb_id", "name_nf", "name_sf", "md5"]

df_companies = pd.read_csv('output/company_name.csv', delimiter=',', names=column_companies, encoding = "ISO-8859-1", low_memory=False)
print(df_companies.dtypes)
print(df_companies.shape)
df_companies = df_companies.drop(columns=["imdb_id", "name_nf", "name_sf", "md5"])
df_companies.head()

company_id    object
name          object
country       object
imdb_id       object
name_nf       object
name_sf       object
md5           object
dtype: object
(5046, 7)


,company_id,name,country
0,ï»¿company_id,name,country
1,289,Ingenious Film Partners,NaN
2,306,Twentieth Century Fox Film Corporation,NaN
3,444,Dune Entertainment,NaN
4,574,Lightstorm Entertainment,NaN


In [373]:
column_keyword = ["keyword_id", "keyword", "phonetic"]

df_keywords = pd.read_csv('output/keyword.csv', delimiter=',', names=column_keyword, encoding = "ISO-8859-1")
print(df_keywords.dtypes)
print(df_keywords.shape)
df_keywords = df_keywords.drop(columns=["phonetic"])
df_keywords.head()

keyword_id    object
keyword       object
phonetic      object
dtype: object
(9804, 3)


,keyword_id,keyword
0,ï»¿keyword_id,keyword
1,1463,culture clash
2,2964,future
3,3386,space war
4,3388,space colony


In [374]:
columns_roles = ["role_id", "cast_role"]

df_roles = pd.read_csv('output/role_type.csv', delimiter=',', names=columns_roles, encoding = "ISO-8859-1")
print(df_roles.dtypes)
print(df_roles.shape)
df_roles.tail()

role_id      object
cast_role    object
dtype: object
(414, 2)


,role_id,cast_role
409,409,Wardrobe Supervisor
410,410,Wig Designer
411,411,Wigmaker
412,412,Writer
413,413,actor


# Merge IMDb and MovieLens

In [375]:
pd.options.display.max_columns = 100

In [376]:
df_movies = pd.merge(df_ML_movies, df_IMDb_movies, on=["title", "release"])
movie_ids = list(df_movies.movie_id.unique())
df_movies.tail()

,movie_id_ml,title,release,url,unknown,action,adventure,animation,childrens,comedy,crime,documentary,drama,fantasy,noir,horror,musical,mystery,romance,scifi,thriller,war,western,movie_id
435,1615,warriors of virtue,1997,http://us.imdb.com/M/title-exact?Warriors%20of...,0,1,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,49478
436,1654,chairman of the board,1998,http://us.imdb.com/Title?Chairman+of+the+Board...,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,31535
437,1664,8 heads in a duffel bag,1997,http://us.imdb.com/Title?8+Heads+in+a+Duffel+B...,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,13982
438,1680,sliding doors,1998,http://us.imdb.com/Title?Sliding+Doors+(1998),0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,10215
439,11,se7en,1995,http://us.imdb.com/M/title-exact?Se7en%20(1995),0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,807


### Create separate DFs

In [377]:
# 1. Ép kiểu cột 'movie_id' ở cả 2 bảng về dạng số nguyên (int64) một cách an toàn
# Sử dụng pd.to_numeric để tự động chuyển đổi, 'coerce' sẽ biến các giá trị lỗi thành NaN
df_movies["movie_id"] = pd.to_numeric(df_movies["movie_id"], errors='coerce')
df_movie_keywords["movie_id"] = pd.to_numeric(df_movie_keywords["movie_id"], errors='coerce')

# 2. Loại bỏ các dòng bị NaN (nếu có) sau khi ép kiểu và chuyển hẳn sang kiểu int64 dữ liệu sạch
df_movies = df_movies.dropna(subset=["movie_id"])
df_movies["movie_id"] = df_movies["movie_id"].astype('int64')

df_movie_keywords = df_movie_keywords.dropna(subset=["movie_id"])
df_movie_keywords["movie_id"] = df_movie_keywords["movie_id"].astype('int64')

# 3. Tiến hành merge bình thường
df_movie = pd.merge(df_movies, df_movie_keywords, on="movie_id")
df_movie = pd.merge(df_movie, df_keywords, on="keyword_id")

# drop excess info
df_movie = df_movie.drop(columns=["movie_id_ml", "keyword_id"])

In [378]:
# 1. Ép kiểu cột 'movie_id' ở cả 2 bảng về dạng số một cách an toàn
# Tham số errors='coerce' giúp biến các chuỗi chữ rác (nếu có) thành NaN thay vì bị crash báo lỗi
df_movies["movie_id"] = pd.to_numeric(df_movies["movie_id"], errors='coerce')
df_movie_keywords["movie_id"] = pd.to_numeric(df_movie_keywords["movie_id"], errors='coerce')

# 2. Loại bỏ các dòng bị khuyết góc (NaN) do ép kiểu lỗi, rồi đưa về dạng số nguyên int64 chuẩn
df_movies = df_movies.dropna(subset=["movie_id"])
df_movies["movie_id"] = df_movies["movie_id"].astype('int64')

df_movie_keywords = df_movie_keywords.dropna(subset=["movie_id"])
df_movie_keywords["movie_id"] = df_movie_keywords["movie_id"].astype('int64')

# 3. Tiến hành gộp (merge) lại - Lúc này chắc chắn sẽ ra đầy đủ dữ liệu
df_movie = pd.merge(df_movies, df_movie_keywords, on="movie_id", how="inner")
df_movie = pd.merge(df_movie, df_keywords, on="keyword_id", how="inner")

# 4. Loại bỏ các cột định danh thừa
df_movie = df_movie.drop(columns=["movie_id_ml", "keyword_id"], errors='ignore')

# Kiểm tra lại kết quả
print(f"Số lượng dòng sau khi merge thành công: {len(df_movie)}")
df_movie.head()

Số lượng dòng sau khi merge thành công: 4450


,title,release,url,unknown,action,adventure,animation,childrens,comedy,crime,documentary,drama,fantasy,noir,horror,musical,mystery,romance,scifi,thriller,war,western,movie_id,keyword
0,toy story,1995,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,862,jealousy
1,toy story,1995,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,862,toy
2,toy story,1995,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,862,boy
3,toy story,1995,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,862,friendship
4,toy story,1995,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,862,friends


In [379]:
df_movie_company = pd.merge(df_movie_companies, df_companies, on="company_id")
df_movie_company.tail()

,movie_id,company_id,note,name,country
13667,9367,5,NaN,Columbia Pictures,NaN
13668,231617,3958,NaN,Front Street Pictures,NaN
13669,231617,6438,NaN,Muse Entertainment Enterprises,NaN
13670,25975,87986,NaN,rusty bear entertainment,NaN
13671,25975,87987,NaN,lucky crow films,NaN


##### Store DFs

In [380]:
import os

# Tự động tạo thư mục 'data' nếu chưa có
os.makedirs("data", exist_ok=True)

# Sau đó chạy các lệnh lưu file bình thường
df_movie.to_csv("data/movies.csv", sep=',', index=False)
df_cast_people.to_csv("data/cast_movie.csv", sep=',', index=False)
df_movie_company.to_csv("data/companies_movie.csv", sep=',', index=False)
df_ratings.to_csv("data/ratings.csv", sep=',', index=False)
df_users.to_csv("data/users.csv", sep=',', index=False)

### Create unique DF (`df`) with everything

##### Keywords

In [381]:
df_movie_keyword = pd.merge(df_movie_keywords[df_movie_keywords.movie_id.isin(movie_ids)], 
                            df_keywords, 
                            on="keyword_id")
df_movie_keyword = df_movie_keyword.groupby("movie_id")["keyword"].apply(list).reset_index()
df_movie_keyword.tail()

,movie_id,keyword
426,61337,"[nun, satire, hospital, half sister]"
427,61752,"[independent film, neighbor, masturbation, pho..."
428,63020,"[immigration, independent film, woman director..."
429,68924,"[based on novel, 1970s, thanksgiving, dysfunct..."
430,87729,"[france, revolution, biography, president, his..."


##### Cast

In [382]:
import pandas as pd
import json

# 1. Đồng bộ kiểu dữ liệu cho tất cả các bảng tham gia
# Sử dụng 'Int64' để giữ lại số nguyên ngay cả khi có giá trị NaN (tránh lỗi IntCastingNaNError)
def clean_id_column(df, col_name):
    if col_name in df.columns:
        # Chuyển về số (lỗi thành NaN), ép về Int64 để mất đuôi .0, rồi đưa về string để merge
        df[col_name] = pd.to_numeric(df[col_name], errors='coerce').astype('Int64').astype(str).str.strip()
        # Loại bỏ các dòng bị biến thành chuỗi "nan" để tránh gộp sai
        return df[df[col_name] != "nan"]
    return df

# Làm sạch các bảng gốc
df_cast = clean_id_column(df_cast, "person_id")
df_cast = clean_id_column(df_cast, "role_id")
df_people = clean_id_column(df_people, "person_id")
df_roles = clean_id_column(df_roles, "role_id")

# 2. Kiểm tra danh sách movie_ids
# Nếu movie_ids bị rỗng, phép lọc .isin() sẽ xóa sạch df_cast
if 'movie_ids' not in locals() or len(movie_ids) == 0:
    print("Cảnh báo: Danh sách movie_ids đang rỗng. Đang lấy toàn bộ phim từ df_cast để tránh bảng trống.")
    df_cast_filtered = df_cast.copy()
else:
    # Đảm bảo movie_id trong df_cast và danh sách movie_ids cùng kiểu dữ liệu
    df_cast["movie_id"] = pd.to_numeric(df_cast["movie_id"], errors='coerce').astype('Int64')
    df_cast_filtered = df_cast[df_cast.movie_id.isin(movie_ids)].copy()

# 3. Thực hiện Merge với LEFT JOIN
# Sử dụng left join để nếu df_people thiếu một vài person_id, dòng đó ở df_cast vẫn được giữ lại
df_movie_cast = pd.merge(df_cast_filtered, df_people, on="person_id", how="left")
df_movie_cast = pd.merge(df_movie_cast, df_roles, on="role_id", how="left")

print(f"Số lượng dòng sau khi gộp: {len(df_movie_cast)}")

# 4. Gom nhóm và chuyển thành JSON (Chỉ chạy nếu có dữ liệu)
if not df_movie_cast.empty:
    # Xác định các cột thực tế đang có trong bảng
    cols_to_json = ["cast_id", "person_id", "cast_name", "cast_gender", "cast_role"]
    valid_cols = [c for c in cols_to_json if c in df_movie_cast.columns]

    df_movie_cast = df_movie_cast.groupby("movie_id")[valid_cols].apply(
        lambda x: json.dumps(list(x.T.to_dict().values()), ensure_ascii=False)
    ).reset_index()

    df_movie_cast.columns = ["movie_id", "cast"]
    print(df_movie_cast.tail())
else:
    print("Lỗi: Bảng vẫn trống sau khi gộp. Hãy kiểm tra lại file df_cast gốc có chứa dữ liệu không.")

Số lượng dòng sau khi gộp: 21084
     movie_id                                               cast
402     61337  [{"cast_id": "163375", "person_id": "13548", "...
403     61752  [{"cast_id": "162853", "person_id": "4604", "c...
404     63020  [{"cast_id": "165574", "person_id": "3141", "c...
405     68924  [{"cast_id": "141881", "person_id": "8945", "c...
406     87729  [{"cast_id": "155394", "person_id": "1733", "c...


##### Companies

In [383]:
import pandas as pd
import json

# --- BƯỚC 1: ĐỒNG BỘ KIỂU DỮ LIỆU KHÓA COMPANY_ID AN TOÀN ---
# Ép kiểu về số, chuyển qua Int64 để khử đuôi .0 rác, cuối cùng đưa về String để gộp
df_movie_companies["company_id"] = pd.to_numeric(df_movie_companies["company_id"], errors='coerce').astype('Int64').astype(str).str.strip()
df_companies["company_id"] = pd.to_numeric(df_companies["company_id"], errors='coerce').astype('Int64').astype(str).str.strip()

# Loại bỏ các dòng bị biến thành chuỗi chữ "nan" sau khi ép kiểu rác
df_movie_companies = df_movie_companies[df_movie_companies["company_id"] != "nan"]
df_companies = df_companies[df_companies["company_id"] != "nan"]


# --- BƯỚC 2: KIỂM TRA VÀ PHÒNG VỆ DANH SÁCH MOVIE_IDS ---
# Nếu danh sách movie_ids bị rỗng, ta bỏ qua bước lọc để không làm rỗng dữ liệu
if 'movie_ids' not in locals() or len(movie_ids) == 0:
    print("⚠️ CẢNH BÁO: Danh sách 'movie_ids' đang bị rỗng (0 dòng) do lỗi gộp phim ở phía trên!")
    print("👉 Đang tự động lấy toàn bộ danh sách phim từ df_movie_companies để cứu dữ liệu bảng này.")
    df_movie_companies_filtered = df_movie_companies.copy()
else:
    # Đồng bộ kiểu dữ liệu cho cột movie_id trước khi dùng lệnh lọc .isin()
    df_movie_companies["movie_id"] = pd.to_numeric(df_movie_companies["movie_id"], errors='coerce').astype('Int64')
    df_movie_companies_filtered = df_movie_companies[df_movie_companies.movie_id.isin(movie_ids)].copy()


# --- BƯỚC 3: THỰC HIỆN GỘP BẰNG LEFT JOIN ---
# Sử dụng how="left" để giữ lại toàn bộ dữ liệu phim, kể cả khi df_companies thiếu thông tin
df_movie_company = pd.merge(df_movie_companies_filtered, df_companies, on="company_id", how="left")

print(f"Số lượng dòng dữ liệu sau khi gộp công ty: {len(df_movie_company)}")


# --- BƯỚC 4: GOM NHÓM VÀ CHUYỂN ĐỔI THÀNH JSON ---
if not df_movie_company.empty:
    # Xác định các cột thực tế đang tồn tại trong bảng để tránh lỗi KeyError
    expected_cols = ["company_id", "name", "country", "note"]
    valid_cols = [col for col in expected_cols if col in df_movie_company.columns]

    # Gom nhóm theo từng movie_id và nén thông tin công ty thành chuỗi JSON
    df_movie_company = df_movie_company.groupby("movie_id")[valid_cols].apply(
        lambda x: json.dumps(list(x.T.to_dict().values()), ensure_ascii=False)
    ).reset_index()

    # Đổi tên cột kết quả một cách an toàn
    df_movie_company.columns = ["movie_id", "company"]

    print("\n--- KẾT QUẢ BẢNG DF_MOVIE_COMPANY ĐÃ CÓ DỮ LIỆU ---")
    print(df_movie_company.tail())
else:
    print("❌ Lỗi nghiêm trọng: Bảng vẫn trống hoàn toàn. Vui lòng kiểm tra lại file df_movie_companies gốc.")

Số lượng dòng dữ liệu sau khi gộp công ty: 974

--- KẾT QUẢ BẢNG DF_MOVIE_COMPANY ĐÃ CÓ DỮ LIỆU ---
     movie_id                                            company
426     61337  [{"company_id": "285", "name": "Live Entertain...
427     61752  [{"company_id": "43", "name": "Fox Searchlight...
428     63020  [{"company_id": "28205", "name": "Samuel Goldw...
429     68924  [{"company_id": "43", "name": "Fox Searchlight...
430     87729  [{"company_id": "2370", "name": "Merchant Ivor...


### Merge 3 above DFs into main movie DF

In [384]:
df = pd.merge(df_movies, df_movie_keyword, on="movie_id")
print(df.shape)
df = pd.merge(df, df_movie_cast, on="movie_id")
print(df.shape)
df = pd.merge(df, df_movie_company, on="movie_id")
df.tail()

(437, 25)
(410, 26)


,movie_id_ml,title,release,url,unknown,action,adventure,animation,childrens,comedy,crime,documentary,drama,fantasy,noir,horror,musical,mystery,romance,scifi,thriller,war,western,movie_id,keyword,cast,company
402,1615,warriors of virtue,1997,http://us.imdb.com/M/title-exact?Warriors%20of...,0,1,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,49478,"[american football, mythology, chinese food, k...","[{""cast_id"": ""98098"", ""person_id"": ""2464"", ""ca...","[{""company_id"": ""2269"", ""name"": ""China Film Co..."
403,1654,chairman of the board,1998,http://us.imdb.com/Title?Chairman+of+the+Board...,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,31535,"[inventor, invention, scandal]","[{""cast_id"": ""180631"", ""person_id"": ""96010"", ""...","[{""company_id"": ""54684"", ""name"": ""101st Street..."
404,1664,8 heads in a duffel bag,1997,http://us.imdb.com/Title?8+Heads+in+a+Duffel+B...,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,13982,"[mexico, vacation, murder, head]","[{""cast_id"": ""197677"", ""person_id"": ""4517"", ""c...","[{""company_id"": ""41"", ""name"": ""Orion Pictures""..."
405,1680,sliding doors,1998,http://us.imdb.com/Title?Sliding+Doors+(1998),0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,10215,"[double life, commercial, fantasy, chance, mar...","[{""cast_id"": ""183122"", ""person_id"": ""12052"", ""...","[{""company_id"": ""4"", ""name"": ""Paramount Pictur..."
406,11,se7en,1995,http://us.imdb.com/M/title-exact?Se7en%20(1995),0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,807,"[self-fulfilling prophecy, detective, s.w.a.t....","[{""cast_id"": ""103431"", ""person_id"": ""287"", ""ca...","[{""company_id"": ""12"", ""name"": ""New Line Cinema..."


##### Store new DFs

In [385]:
df.to_csv("data/transformid.csv", encoding="utf8")

---

# Read new DFs

In [386]:
import pandas as pd

In [387]:
df_movies_all = pd.read_csv("data/movies.csv", sep=',')
df_cast_people = pd.read_csv("data/cast_movie.csv", sep=',')
df_movie_companies = pd.read_csv("data/companies_movie.csv", sep=',')
df_ratings = pd.read_csv("data/ratings.csv", sep=',')
df_users = pd.read_csv("data/users.csv", sep=',')

C:\Users\ACER\AppData\Local\Temp\ipykernel_6736\183546395.py:2: DtypeWarning: Columns (0,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_cast_people = pd.read_csv("data/cast.csv", sep=',')


In [388]:
import pandas_profiling

In [389]:
if df_movies_all.empty:
    print("CẢNH BÁO: DataFrame đang trống rỗng! Hãy kiểm tra lại các bước gộp dữ liệu phía trên.")
else:
    import os
    os.makedirs("data", exist_ok=True) # Đảm bảo có thư mục data như bước trước

    profile_movies = df_movies_all.profile_report(title='Movies Report')
    profile_movies.to_file(output_file="data/movies_report.html")
    print("Đã xuất báo cáo thành công!")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 65.06it/s]

Đã xuất báo cáo thành công!


In [390]:
import pandas as pd

# 1. Làm sạch df_cast: Loại bỏ các dòng trống ở cột khóa ban đầu (nếu có)
df_cast = df_cast.dropna(subset=["person_id", "role_id"])

# 2. Ép kiểu khóa an toàn cho df_cast (Sử dụng 'Int64' viết hoa để xử lý lỗi NaN)
# Bước này biến "413.0" hoặc các dòng rác thành số nguyên 413, sau đó mới đưa về chuỗi "413" sạch rác
df_cast["person_id"] = pd.to_numeric(df_cast["person_id"], errors='coerce').astype('Int64').astype(str).str.strip()
df_cast["role_id"] = pd.to_numeric(df_cast["role_id"], errors='coerce').astype('Int64').astype(str).str.strip()

# Loại bỏ các dòng phát sinh chuỗi chữ "nan" sau khi ép kiểu để tránh làm rỗng bảng lúc merge
df_cast = df_cast[(df_cast["person_id"] != "nan") & (df_cast["role_id"] != "nan")]

# 3. Ép kiểu tương tự cho df_people và df_roles để đảm bảo đồng bộ kiểu dữ liệu chuỗi khi merge
df_people["person_id"] = pd.to_numeric(df_people["person_id"], errors='coerce').astype('Int64').astype(str).str.strip()
df_people = df_people[df_people["person_id"] != "nan"]

df_roles["role_id"] = pd.to_numeric(df_roles["role_id"], errors='coerce').astype('Int64').astype(str).str.strip()
df_roles = df_roles[df_roles["role_id"] != "nan"]

# 4. Thực hiện gộp dữ liệu bằng LEFT JOIN (how="left") thay vì Inner Join mặc định.
# Cách này giúp giữ lại toàn bộ danh sách diễn viên ở bảng df_cast gốc, tránh việc lệch một vài mã làm mất sạch dữ liệu.
df_cast_people = pd.merge(df_cast, df_people, on="person_id", how="left")
df_cast_people = pd.merge(df_cast_people, df_roles, on="role_id", how="left")

# 5. Xóa cột thừa và kiểm tra lại số lượng dòng dữ liệu
df_cast_people = df_cast_people.drop(columns=["role_id"], errors='ignore')
print(f"Số lượng dòng hiện tại của df_cast_people là: {len(df_cast_people)}")

# 6. Chỉ tiến hành chạy báo cáo ydata_profiling nếu dữ liệu thực sự tồn tại dòng
if not df_cast_people.empty and len(df_cast_people) > 0:
    # Đảm bảo import thư viện nếu chưa khai báo ở đầu file
    from ydata_profiling import ProfileReport

    profile_cast = df_cast_people.profile_report(title='Cast Report')
    profile_cast.to_file(output_file="data/cast_report.html")
    print("Đã xuất báo cáo Cast Report thành công tại thư mục 'data/cast_report.html'!")
else:
    print("Cảnh báo: Bảng df_cast_people đang trống (0 dòng)! Vui lòng kiểm tra lại sự trùng khớp ID giữa các file CSV đầu vào.")

Số lượng dòng hiện tại của df_cast_people là: 217692


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 119.05it/s]

Đã xuất báo cáo Cast Report thành công tại thư mục 'data/cast_report.html'!


In [391]:
profile_companies = df_movie_companies.profile_report(title='Companies Report')
profile_companies.to_file(output_file="data/companies_report.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 246.68it/s]


In [392]:
profile_ratings = df_ratings.profile_report(title='Ratings Report')
profile_ratings.to_file(output_file="data/ratings_report.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 247.70it/s]


In [393]:
profile_users = df_users.profile_report(title='Users Report')
profile_users.to_file(output_file="data/users_report.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 170.28it/s]
